In [1]:
import cadquery as cq
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Moldings", cad_width=800, height=600)
set_default_viewer("Moldings")

In [3]:
def bevel_weatherboard(top_width, bottom_width, height):

    profile_points = []

    # Add initial points
    profile_points.append((0,0))
    profile_points.append((top_width,0))
    profile_points.append((bottom_width, -height))
    profile_points.append((0, -height))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile
    

In [4]:
def beaded_weatherboard(top_width, bottom_width, height):

    profile_points = []
    segments = 32
    increment = 180/segments

    # Add initial points
    profile_points.append((0,0))
    profile_points.append((top_width,0))

    # Define the bead
    bead_diameter = bottom_width
    bead_radius = bead_diameter / 2
    bevel_height = height - bead_diameter
    bevel_width = bottom_width * 0.90
    center_x = bead_radius
    center_y = -height + bead_radius

    # Add the stopping point before the bead
    profile_points.append((bevel_width, -bevel_height))
    profile_points.append((center_x, -bevel_height))

    # Add the bead points from 90 to 270 degrees
    for segment in range(1,segments + 1):

        if segment <= (segments/2):
            angle_degrees = 90 - (segment * increment)
        else:
            segment_counter = segment - (segments/2)
            angle_degrees = 360 - (segment_counter * increment)

        angle_radians = math.radians(angle_degrees)
        
        bead_x = center_x + (bead_radius * math.cos(angle_radians))
        bead_y = center_y + (bead_radius * math.sin(angle_radians))

        profile_points.append((bead_x, bead_y))

    # Add the final point
    profile_points.append((0, -height))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile
    

In [5]:
def beaded_board(width, height):

    profile_points = []
    segments = 32
    increment = 180/segments

    # Define the bead
    bead_diameter = width
    bead_radius = bead_diameter / 2
    board_height = height - bead_diameter
    board_width = width
    center_x = bead_radius
    center_y = -height + bead_radius

    # Add initial point
    profile_points.append((0,0))
    profile_points.append((board_width,0))
    profile_points.append((board_width, -board_height))

    # Add the bead points from 90 to 270 degrees
    for segment in range(1,segments):

        if segment <= (segments/2):
            angle_degrees = 90 - (segment * increment)
        else:
            segment_counter = segment - (segments/2)
            angle_degrees = 360 - (segment_counter * increment)

        angle_radians = math.radians(angle_degrees)
        
        bead_x = center_x + (bead_radius * math.cos(angle_radians))
        bead_y = center_y + (bead_radius * math.sin(angle_radians))

        profile_points.append((bead_x, bead_y))

    # Add the board corners
    profile_points.append((bead_radius, -height))
    profile_points.append((0, -height))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [6]:
def ovolo_muntin(
    width: float = 1.25,
    height: float = 1.0,
    nose_width: float = 0.25,
    nose_height: float = 0.125,
    wing_width: float = 0.125,
    wing_height: float = 0.25,
    stem_width: float = 0.5,
    stem_height: float = 0.25
) -> cq.Workplane():

    segments = 32
    increment = 90/segments
    ovolo_radius = width/2 - ((nose_width/2) + wing_width)
    profile_points = []
    bottom_width = (width - stem_width)/2

    # Left wing
    profile_points.append((0,-wing_height))
    profile_points.append((0,0))
    profile_points.append((wing_width,0))

    # Left arc parameters
    center_x = width/2 - (nose_width/2)
    center_y = 0
    
    # Left arc
    for segment in range(1,segments):

        angle_degrees = 180 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = center_x + (ovolo_radius * math.cos(angle_radians))
        arc_y = center_y + (ovolo_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Nose
    profile_points.append((width/2 - nose_width/2, ovolo_radius))
    profile_points.append((width/2 - nose_width/2, ovolo_radius + nose_height))
    profile_points.append((width/2 + nose_width/2, ovolo_radius + nose_height))
    profile_points.append((width/2 + nose_width/2, ovolo_radius))

    # Right arc parameters
    center_x = width/2 + (nose_width/2)
    center_y = 0
    
    # Right arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = center_x + (ovolo_radius * math.cos(angle_radians))
        arc_y = center_y + (ovolo_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Right wing
    profile_points.append((width - wing_width, 0))
    profile_points.append((width, 0))
    profile_points.append((width, -wing_height))

    # Stem
    profile_points.append((width - bottom_width, -wing_height))
    profile_points.append((width - bottom_width, -wing_height - stem_height))
    profile_points.append((bottom_width, -wing_height - stem_height))
    profile_points.append((bottom_width, -wing_height))

    # Close the loop
    profile_points.append((0,-wing_height))
    
    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile
    

In [7]:
def crown_molding(width, height):

    segments = 32
    increment = 90/segments
    board_height = height / math.sqrt(2)
    fillet_height = (1/9) * board_height
    board_width = fillet_height
    profile_points = []

    # Top fillet
    profile_points.append((board_width,0))
    profile_points.append((0,0))
    profile_points.append((0,-board_width))

    # Cyma recta parameters
    cyma_recta_height = (4/9) * board_height
    cyma_recta_radius = cyma_recta_height / 2

    # Cyma recta concave parameters
    concave_center_x = 0
    concave_center_y = -board_width - cyma_recta_radius

    # Cyma recta concave arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = concave_center_x + (cyma_recta_radius * math.cos(angle_radians))
        arc_y = concave_center_y + (cyma_recta_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Cyma recta convex parameters
    convex_center_x = 0 + (2 * cyma_recta_radius)
    convex_center_y = concave_center_y

    # Cyma recta convex arc
    for segment in range(1,segments):

        angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (cyma_recta_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (cyma_recta_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Middle fillet
    profile_points.append((convex_center_x, convex_center_y - cyma_recta_radius))
    profile_points.append((convex_center_x, convex_center_y - cyma_recta_radius - fillet_height))
    profile_points.append((convex_center_x + fillet_height, convex_center_y - cyma_recta_radius - fillet_height))

    # Cyma reversa parameters
    cyma_reversa_height = (2/9) * board_height
    cyma_reversa_radius = cyma_reversa_height / 2
    
    # Cyma reversa convex parameters
    convex_center_x = convex_center_x + fillet_height + cyma_reversa_radius
    convex_center_y = convex_center_y - cyma_recta_radius - fillet_height

    # Cyma reversa convex arc
    for segment in range(1,segments):

        angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Cyma reversa concave parameters
    concave_center_x = convex_center_x
    concave_center_y = convex_center_y - (cyma_reversa_radius * 2)
    
    # Cyma reversa concave arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = concave_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = concave_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Add the bottom fillet
    profile_points.append((concave_center_x + cyma_reversa_radius, concave_center_y))
    profile_points.append((concave_center_x + cyma_reversa_radius + board_width, concave_center_y))
    profile_points.append((concave_center_x + cyma_reversa_radius + board_width, concave_center_y + board_width))
        
    # Create the polyline
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

    

In [8]:
def cavetto_board(
    offset_x: float = 0.0,
    offset_y: float = 0.0,
    top_width: float = 1.0,
    bottom_width: float = 0.25,
    height: float = 8.0
) -> cq.Workplane:

    segments = 32
    increment = 90/segments
    radius = (top_width - bottom_width)
    profile_points = []

    # Append starting points
    profile_points.append((offset_x, offset_y))
    profile_points.append((offset_x + top_width, offset_y))
    profile_points.append((offset_x + top_width, offset_y - height))
    profile_points.append((offset_x + radius, offset_y - height))

    # Draw the arc
    center_x = offset_x
    center_y = offset_y - height

    for segment in range(1,segments):

        angle_degrees = 0 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = center_x + (radius * math.cos(angle_radians))
        arc_y = center_y + (radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))


    # Add the final point
    profile_points.append((center_x, center_y + radius))

    # Create the polyline
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [9]:
def bed_molding(width, height):

    segments = 32
    increment = 90/segments
    board_height = height / math.sqrt(2)
    fillet_height = (1/9) * board_height
    board_width = fillet_height
    profile_points = []

    # Top fillet
    profile_points.append((board_width,0))
    profile_points.append((0,0))
    profile_points.append((0,-board_width))
    profile_points.append((board_width,-board_width))

    # Ovolo parameters
    ovolo_height = (4/9) * board_height
    ovolo_radius = ovolo_height / 2

    # Ovolo convex parameters
    convex_center_x = board_width + ovolo_radius
    convex_center_y = -board_width

    # Cyma recta convex arc
    for segment in range(1,segments):

        angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (ovolo_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (ovolo_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Middle fillet
    profile_points.append((convex_center_x, convex_center_y - ovolo_radius))
    profile_points.append((convex_center_x, convex_center_y - ovolo_radius - fillet_height))
    profile_points.append((convex_center_x + fillet_height, convex_center_y - ovolo_radius - fillet_height))

    # Cyma reversa parameters
    cyma_reversa_height = (2/9) * board_height
    cyma_reversa_radius = cyma_reversa_height / 2
    
    # Cyma reversa convex parameters
    convex_center_x = convex_center_x + fillet_height + cyma_reversa_radius
    convex_center_y = convex_center_y - ovolo_radius - fillet_height

    # Cyma reversa convex arc
    for segment in range(1,segments):

        angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Cyma reversa concave parameters
    concave_center_x = convex_center_x
    concave_center_y = convex_center_y - (cyma_reversa_radius * 2)
    
    # Cyma reversa concave arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = concave_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = concave_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Add the bottom fillet
    profile_points.append((concave_center_x + cyma_reversa_radius, concave_center_y))
    profile_points.append((concave_center_x + cyma_reversa_radius + board_width, concave_center_y))
    profile_points.append((concave_center_x + cyma_reversa_radius + board_width, concave_center_y + board_width))
        
    # Create the polyline
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [10]:
def cyma_reversa_band(width, height):

    segments = 32
    increment = 90/segments
    profile_points = []
    fillet_height = (1/3) * height

    # Add initial points
    profile_points.append((width,fillet_height))
    profile_points.append((-fillet_height,fillet_height))
    profile_points.append((-fillet_height,0))
    profile_points.append((0,0))
    profile_points.append((width,0))
    
    
    
    # Cyma reversa parameters
    cyma_reversa_height = (2/3) * height
    cyma_reversa_radius = cyma_reversa_height / 2
    
    # Cyma reversa convex parameters
    convex_center_x = cyma_reversa_radius
    convex_center_y = 0

    # Cyma reversa convex arc
    for segment in range(1,segments):

        angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Cyma reversa concave parameters
    concave_center_x = convex_center_x
    concave_center_y = convex_center_y - (cyma_reversa_radius * 2)
    
    # Cyma reversa concave arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = concave_center_x + (cyma_reversa_radius * math.cos(angle_radians))
        arc_y = concave_center_y + (cyma_reversa_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Add the final points
    profile_points.append((cyma_reversa_radius * 2,-height))
    profile_points.append((width,-height))

    # Create the polyline
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [11]:
def georgian_cornice(length):

    # Create the cornice assembly
    cornice = cq.Assembly()
    
    # Crown
    crown = crown_molding(1, 6).extrude(length).translate((4.25,0,6))
    cornice.add(crown)
    
    # Corona
    cavetto = cavetto_board().extrude(length).translate((8,0,6)).rotateAboutCenter((0,0,1),180)
    fascia = cq.Workplane("XZ").rect(10,0.75).extrude(length).translate((14,0,-0.8))
    cornice.add(cavetto)
    cornice.add(fascia)
    
    # Modillion
    modillion_count = math.floor(length/9)
    modillion_backing = cq.Workplane("XZ").rect(4.5,0.75).extrude(length).translate((19,0,-3.5)).rotateAboutCenter((0,1,0),90)
    cornice.add(modillion_backing)
    
    for modillion in range(0,modillion_count):
    
        modillion_x = 240 - (modillion * 9)
        modillion = cq.Workplane("XZ").rect(5.5,3.0).extrude(3).translate((16.25,-modillion_x + 9,-4)) # old value 16.25
        modillion_band = cyma_reversa_band(7,1).extrude(3).translate((12,-modillion_x + 9,-1.5)) # old value 12
        cornice.add(modillion)
        cornice.add(modillion_band)
    
    
    # Bedmold 
    bed = bed_molding(0.75,5.5).extrude(length).translate((18,0,-5.5))
    cornice.add(bed)

    return cornice

In [12]:
cornice = georgian_cornice(240)

show(cornice)

+++++++++++++++++++++++++++++++++++++++++++++++++++++++++
